# The birthday paradox, and hash-table collisions

With $d$ equally likely birthdays and $n$ people, the probability that all $n$ are distinct is

$$P(\text{all distinct}) = \prod_{i=0}^{n-1}\frac{d-i}{d} = \frac{d!}{d^{\,n}\,(d-n)!},$$

so the probability that at least two coincide is

$$P(\text{collision}) = 1 - \prod_{i=0}^{n-1}\frac{d-i}{d}.$$

For $d = 365$ this crosses $\tfrac{1}{2}$ at just $n = 23$ — far fewer than intuition suggests, because what matters is the number of *pairs*, $\binom{n}{2} = \tfrac{n(n-1)}{2}$, which grows quadratically in $n$.

A hash table is the same problem with $d = m$: dropping $n$ keys into $m$ buckets, the expected number of colliding pairs is $\binom{n}{2}/m$. Collisions therefore appear long before the table is full.

## A minimal hash map

Separate chaining: $m$ buckets, each a list. A universal hash spreads keys evenly,

$$h(k) = \big((a\,k + b) \bmod p\big) \bmod m,$$

with $a, b$ chosen at random and $p$ the Mersenne prime $2^{61}-1$. When the load factor $n/m$ passes the table's `load_factor`, it grows to the next prime in the list and rehashes every key, keeping chains short — which is why the bucket counts are a *list* of primes, not a single number. The class counter `comparisons` tallies every key comparison a chain walk makes; that is the price collisions charge.

In [1]:
import random

PRIMES = [3, 7, 13, 31, 61, 127, 251, 509, 1021, 2039, 4093, 8191]
MERSENNE = (1 << 61) - 1


class HashMap:
    comparisons = 0

    def __init__(self, n_buckets=PRIMES[0], load_factor=0.75, grow=True):
        self.m = n_buckets
        self.load_factor = load_factor
        self.grow = grow
        self.n = 0
        self.buckets = [[] for _ in range(self.m)]
        self.a = random.randrange(1, MERSENNE)
        self.b = random.randrange(0, MERSENNE)

    def _index(self, key):
        return ((self.a * key + self.b) % MERSENNE) % self.m

    def put(self, key, value):
        chain = self.buckets[self._index(key)]
        for i, (k, _) in enumerate(chain):
            HashMap.comparisons += 1
            if k == key:
                chain[i] = (key, value)
                return
        chain.append((key, value))
        self.n += 1
        if self.grow and self.n > self.load_factor * self.m:
            self._resize()

    def get(self, key):
        for k, v in self.buckets[self._index(key)]:
            HashMap.comparisons += 1
            if k == key:
                return v
        raise KeyError(key)

    def _resize(self):
        bigger = next((p for p in PRIMES if p > self.m), None)
        if bigger is None:
            return
        self.m = bigger
        items = [pair for chain in self.buckets for pair in chain]
        self.buckets = [[] for _ in range(self.m)]
        for k, v in items:
            self.buckets[self._index(k)].append((k, v))

## What collisions cost

Turn resizing off (`grow=False`) and fix the table at $m$ buckets. Insert $n$ keys and count every comparison the chain walks make: that running total lands right on $\binom{n}{2}/m$ — the expected number of colliding pairs, the exact quantity from the birthday paradox above. It grows with the *square* of $n$.

So the work per insert rises with the load factor $\alpha = n/m$: doubling $\alpha$ roughly quadruples the total comparisons. Drag the slider to read the total at a given load factor — at $\alpha = 0.5$ it is modest, at $\alpha = 1$ about four times as much. That is precisely why the real map refuses to let $\alpha$ climb: it resizes once $n/m$ passes `load_factor`, trading a rehash now for short chains — and $O(1)$ lookups — later.

In [2]:
import plotly.graph_objects as go

random.seed(0)
m = PRIMES[8]
trials = 40
totals = [0.0] * (m + 1)
for _ in range(trials):
    hm = HashMap(n_buckets=m, grow=False)
    HashMap.comparisons = 0
    for i, key in enumerate(random.sample(range(10_000_000), m), 1):
        hm.put(key, None)
        totals[i] += HashMap.comparisons
avg = [t / trials for t in totals]
ns = list(range(m + 1))
expected = [n * (n - 1) / 2 / m for n in ns]
load_factors = [round(0.1 * t, 1) for t in range(1, 11)]

fig = go.Figure()
fig.add_trace(go.Scatter(x=ns, y=avg, mode="lines", name="measured comparisons",
                         line=dict(color="#1f77b4")))
fig.add_trace(go.Scatter(x=ns, y=expected, mode="lines", name="C(n, 2) / m",
                         line=dict(color="#999999", dash="dash")))


def frame(alpha):
    na = round(alpha * m)
    ca = avg[na]
    return dict(
        shapes=[dict(type="line", x0=na, x1=na, y0=0, y1=ca,
                     line=dict(color="crimson", dash="dot"))],
        annotations=[dict(x=na, y=ca, ax=-45, ay=-30, showarrow=True, arrowhead=2,
                          font=dict(color="crimson"),
                          text=f"load factor {alpha}: {ca:.0f} comparisons")],
        title=f"Insertion comparisons in a fixed table of m = {m} buckets (load factor {alpha})",
    )


steps = [dict(method="relayout", args=[frame(alpha)], label=str(alpha))
         for alpha in load_factors]
init = frame(1.0)
fig.update_layout(
    sliders=[dict(active=len(load_factors) - 1,
                  currentvalue={"prefix": "load factor = "}, steps=steps)],
    shapes=init["shapes"], annotations=init["annotations"], title=init["title"],
    xaxis_title="keys inserted (n)",
    yaxis_title="total key comparisons",
    margin=dict(t=60),
)
fig.show()

**[In-browser notebook →](/lite/notebooks/index.html?path=2026-09-24-birthday-paradox-and-hashing.ipynb)**